In [123]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from scipy.stats import chi2_contingency
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_recall_fscore_support
import warnings
import os

In [124]:
a1 = pd.read_excel("datasets/case_study1.xlsx")
a2 = pd.read_excel("datasets/case_study2.xlsx")

In [125]:
df1 = a1.copy()
df2 = a2.copy()

In [126]:
df2.shape

(51336, 62)

In [127]:
df1 = df1.loc[df1['Age_Oldest_TL'] != -99999]  #removing the rows that have age_oldestTl = -99999

In [128]:
df1.shape       #if there are null values more than 10k then we will drop that clumn and if there are less then it then we will drop rows

(51296, 26)

In [129]:
df2.shape

(51336, 62)

In [130]:
columns_to_be_removed = []
for i in df2.columns:
    if df2.loc[df2[i] == -99999].shape[0]>10000:
        columns_to_be_removed.append(i)

In [131]:
df2 = df2.drop(columns_to_be_removed, axis=1)

In [132]:
df2.shape

(51336, 54)

In [133]:
for i in df2.columns:
    df2 = df2.loc[df2[i] != -99999]

In [134]:
df2.shape

(42066, 54)

In [135]:
for i in list(df1.columns):     # checking if there are same cloumns in both tables
    if i in list(df2.columns):      # jo common hai uske help se join kr denge
        print(i)

PROSPECTID


In [136]:
df2.shape

(42066, 54)

In [137]:
df = pd.merge(df1, df2, how = 'inner' , left_on =['PROSPECTID'], right_on =['PROSPECTID'])

In [138]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 79 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PROSPECTID                  42064 non-null  int64  
 1   Total_TL                    42064 non-null  int64  
 2   Tot_Closed_TL               42064 non-null  int64  
 3   Tot_Active_TL               42064 non-null  int64  
 4   Total_TL_opened_L6M         42064 non-null  int64  
 5   Tot_TL_closed_L6M           42064 non-null  int64  
 6   pct_tl_open_L6M             42064 non-null  float64
 7   pct_tl_closed_L6M           42064 non-null  float64
 8   pct_active_tl               42064 non-null  float64
 9   pct_closed_tl               42064 non-null  float64
 10  Total_TL_opened_L12M        42064 non-null  int64  
 11  Tot_TL_closed_L12M          42064 non-null  int64  
 12  pct_tl_open_L12M            42064 non-null  float64
 13  pct_tl_closed_L12M          420

In [139]:
for i in df.columns:        #checking categorical data
    if df[i].dtype == 'object':
        print(i)

MARITALSTATUS
EDUCATION
GENDER
last_prod_enq2
first_prod_enq2
Approved_Flag


# chi-square test

In [140]:
for i in ['MARITALSTATUS','EDUCATION','GENDER','last_prod_enq2','first_prod_enq2','Approved_Flag']:
    chi2, pval, _, _ = chi2_contingency(pd.crosstab(df[i], df['Approved_Flag']))
    print(i, '---',pval)        #pvalues of those columns having less then 0.05 will be 'failed to reject' means accepted

MARITALSTATUS --- 3.578180861038862e-233
EDUCATION --- 2.6942265249737532e-30
GENDER --- 1.907936100186563e-05
last_prod_enq2 --- 0.0
first_prod_enq2 --- 7.84997610555419e-287
Approved_Flag --- 0.0


# VIF for numerical columns

In [141]:
numeric_columns = []
for i in df.columns:
    if df[i].dtype != 'object' and i not in ['PROSPECTID','Approved_Flag']:
        numeric_columns.append(i)

In [142]:
numeric_columns

['Total_TL',
 'Tot_Closed_TL',
 'Tot_Active_TL',
 'Total_TL_opened_L6M',
 'Tot_TL_closed_L6M',
 'pct_tl_open_L6M',
 'pct_tl_closed_L6M',
 'pct_active_tl',
 'pct_closed_tl',
 'Total_TL_opened_L12M',
 'Tot_TL_closed_L12M',
 'pct_tl_open_L12M',
 'pct_tl_closed_L12M',
 'Tot_Missed_Pmnt',
 'Auto_TL',
 'CC_TL',
 'Consumer_TL',
 'Gold_TL',
 'Home_TL',
 'PL_TL',
 'Secured_TL',
 'Unsecured_TL',
 'Other_TL',
 'Age_Oldest_TL',
 'Age_Newest_TL',
 'time_since_recent_payment',
 'num_times_delinquent',
 'max_recent_level_of_deliq',
 'num_deliq_6mts',
 'num_deliq_12mts',
 'num_deliq_6_12mts',
 'num_times_30p_dpd',
 'num_times_60p_dpd',
 'num_std',
 'num_std_6mts',
 'num_std_12mts',
 'num_sub',
 'num_sub_6mts',
 'num_sub_12mts',
 'num_dbt',
 'num_dbt_6mts',
 'num_dbt_12mts',
 'num_lss',
 'num_lss_6mts',
 'num_lss_12mts',
 'recent_level_of_deliq',
 'tot_enq',
 'CC_enq',
 'CC_enq_L6m',
 'CC_enq_L12m',
 'PL_enq',
 'PL_enq_L6m',
 'PL_enq_L12m',
 'time_since_recent_enq',
 'enq_L12m',
 'enq_L6m',
 'enq_L3m',

# VIF sequentially check

In [147]:
vif_data = df[numeric_columns]
total_columns = vif_data.shape[1]
columns_to_be_kept = []
column_index = 0



for i in range (0,total_columns):
    
    vif_value = variance_inflation_factor(vif_data, column_index)
    print (column_index,'---',vif_value)
    
    
    if vif_value <= 6:
        columns_to_be_kept.append( numeric_columns[i] )
        column_index = column_index+1
    
    else:
        vif_data = vif_data.drop([ numeric_columns[i] ] , axis=1)


C:\Users\hp\AppData\Local\Temp\ipykernel_5260\4277178579.py:10: UserWarning: The design matrix is poorly conditioned (condition number=2.56e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)
C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\hp\AppData\Local\Temp\ipykernel_5260\4277178579.py:10: UserWarning: The design matrix is poorly conditioned (condition number=1.71e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)


0 --- 17519666.49879123


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\hp\AppData\Local\Temp\ipykernel_5260\4277178579.py:10: UserWarning: The design matrix is poorly conditioned (condition number=2.07e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)


0 --- 41463544.73508846


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\hp\AppData\Local\Temp\ipykernel_5260\4277178579.py:10: UserWarning: The design matrix is poorly conditioned (condition number=2.02e+16). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)


...

15 --- 2.7209536161278254


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
C:\Users\hp\AppData\Local\Temp\ipykernel_5260\4277178579.py:10: UserWarning: The design matrix is poorly conditioned (condition number=3.78e+15). VIF calculations may be numerically unstable.
  vif_value = variance_inflation_factor(vif_data, column_index)


16 --- 6.909958748540417


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared


16 --- 4.921475721780697
17 --- 1000799917193443.5
17 --- 6.64060783017616
17 --- 1.3159154707874672
18 --- 7.625456477063671
18 --- 1.5813808513855974
19 --- 6.247287793745863
19 --- 13.46118303405988
19 --- 2.115309430781643
20 --- 1.4995355193869966
21 --- 2.1707974597343567
22 --- 2.6194659619474265
23 --- 2.2938993517650523
24 --- 7.358623292167633
24 --- 2.159662759661746
25 --- 2.866271116689627
26 --- 6.456924734738711
26 --- 2.84711095386759
27 --- 4.498908692151739
28 --- 9.682605341254286
28 --- 5.967208313487481
29 --- 8.495827332703996
29 --- 5.152084361610737
30 --- 7.280751669212374
30 --- 11.481328638663093
30 --- 2.9675076428961265
31 --- 1.595005990346598
32 --- 12.53092282701146
32 --- 7.836355832427298
32 --- 3.0668746024519655
33 --- 1.4041913778107664
34 --- 1.0484933703163495
35 --- 1.2675679133867637
36 --- 3.1270620692109894
37 --- 4.340308450459915
38 --- 1.0008022071023928
39 --- 2.8456129815467026
40 --- 2.2794120029683613
41 --- 15.827331223869045
41 --- 16

In [148]:
print("Original numeric columns:", len(numeric_columns))
print("Final columns:", len(columns_to_be_kept))
print(columns_to_be_kept)

Original numeric columns: 72
Final columns: 46
['pct_tl_open_L6M', 'pct_tl_closed_L6M', 'Total_TL_opened_L12M', 'Tot_TL_closed_L12M', 'pct_tl_open_L12M', 'pct_tl_closed_L12M', 'Tot_Missed_Pmnt', 'CC_TL', 'Home_TL', 'PL_TL', 'Secured_TL', 'Unsecured_TL', 'Other_TL', 'Age_Oldest_TL', 'Age_Newest_TL', 'time_since_recent_payment', 'max_recent_level_of_deliq', 'num_deliq_6_12mts', 'num_times_60p_dpd', 'num_std_12mts', 'num_sub', 'num_sub_6mts', 'num_sub_12mts', 'num_dbt', 'num_dbt_12mts', 'num_lss', 'num_lss_12mts', 'recent_level_of_deliq', 'CC_enq', 'CC_enq_L12m', 'PL_enq_L12m', 'time_since_recent_enq', 'enq_L3m', 'AGE', 'NETMONTHLYINCOME', 'Time_With_Curr_Empr', 'pct_of_active_TLs_ever', 'pct_opened_TLs_L6m_of_L12m', 'pct_currentBal_all_TL', 'CC_Flag', 'PL_Flag', 'pct_PL_enq_L6m_of_ever', 'pct_CC_enq_L6m_of_ever', 'HL_Flag', 'GL_Flag', 'Credit_Score']


# check Anova for columns_to_be_kept 

In [149]:

from scipy.stats import f_oneway

columns_to_be_kept_numerical = []

for i in columns_to_be_kept:
    a = list(df[i])  
    b = list(df['Approved_Flag'])  
    
    group_P1 = [value for value, group in zip(a, b) if group == 'P1']
    group_P2 = [value for value, group in zip(a, b) if group == 'P2']
    group_P3 = [value for value, group in zip(a, b) if group == 'P3']
    group_P4 = [value for value, group in zip(a, b) if group == 'P4']


    f_statistic, p_value = f_oneway(group_P1, group_P2, group_P3, group_P4)

    if p_value <= 0.05:
        columns_to_be_kept_numerical.append(i)



In [150]:
print(len(columns_to_be_kept_numerical))

44


# feature selection is done

# label encoding for the categorical features

In [151]:
features = columns_to_be_kept_numerical + ['MARITALSTATUS', 'EDUCATION', 'GENDER', 'last_prod_enq2', 'first_prod_enq2']
df = df[features+['Approved_Flag']]

In [153]:
df['MARITALSTATUS'].unique()    

array(['Married', 'Single'], dtype=object)

In [154]:

df['EDUCATION'].unique()


array(['12TH', 'GRADUATE', 'SSC', 'POST-GRADUATE', 'UNDER GRADUATE',
       'OTHERS', 'PROFESSIONAL'], dtype=object)

In [155]:

df['GENDER'].unique()


array(['M', 'F'], dtype=object)

In [156]:

df['last_prod_enq2'].unique()


array(['PL', 'ConsumerLoan', 'AL', 'CC', 'others', 'HL'], dtype=object)

In [157]:

df['first_prod_enq2'].unique()

array(['PL', 'ConsumerLoan', 'others', 'AL', 'HL', 'CC'], dtype=object)

In [158]:
df.loc[df['EDUCATION'] == 'SSC',['EDUCATION']]              = 1         #assigning values to education column only , for other we can do onehot encoding
df.loc[df['EDUCATION'] == '12TH',['EDUCATION']]             = 2
df.loc[df['EDUCATION'] == 'GRADUATE',['EDUCATION']]         = 3
df.loc[df['EDUCATION'] == 'UNDER GRADUATE',['EDUCATION']]   = 3
df.loc[df['EDUCATION'] == 'POST-GRADUATE',['EDUCATION']]    = 4
df.loc[df['EDUCATION'] == 'OTHERS',['EDUCATION']]           = 1
df.loc[df['EDUCATION'] == 'PROFESSIONAL',['EDUCATION']]     = 3

In [160]:
df['EDUCATION'].value_counts()

EDUCATION
3    18931
2    11703
1     9532
4     1898
Name: count, dtype: int64

In [162]:

df['EDUCATION'] = df['EDUCATION'].astype(int)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 50 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   pct_tl_open_L6M             42064 non-null  float64
 1   pct_tl_closed_L6M           42064 non-null  float64
 2   Total_TL_opened_L12M        42064 non-null  int64  
 3   Tot_TL_closed_L12M          42064 non-null  int64  
 4   pct_tl_open_L12M            42064 non-null  float64
 5   pct_tl_closed_L12M          42064 non-null  float64
 6   Tot_Missed_Pmnt             42064 non-null  int64  
 7   CC_TL                       42064 non-null  int64  
 8   Home_TL                     42064 non-null  int64  
 9   PL_TL                       42064 non-null  int64  
 10  Secured_TL                  42064 non-null  int64  
 11  Unsecured_TL                42064 non-null  int64  
 12  Other_TL                    42064 non-null  int64  
 13  Age_Oldest_TL               420

In [167]:
df_encoded = pd.get_dummies(
    df,
    columns=['MARITALSTATUS','GENDER','last_prod_enq2','first_prod_enq2'],
    dtype='uint8'
)

In [168]:
df_encoded.info()
k = df_encoded.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42064 entries, 0 to 42063
Data columns (total 62 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   pct_tl_open_L6M               42064 non-null  float64
 1   pct_tl_closed_L6M             42064 non-null  float64
 2   Total_TL_opened_L12M          42064 non-null  int64  
 3   Tot_TL_closed_L12M            42064 non-null  int64  
 4   pct_tl_open_L12M              42064 non-null  float64
 5   pct_tl_closed_L12M            42064 non-null  float64
 6   Tot_Missed_Pmnt               42064 non-null  int64  
 7   CC_TL                         42064 non-null  int64  
 8   Home_TL                       42064 non-null  int64  
 9   PL_TL                         42064 non-null  int64  
 10  Secured_TL                    42064 non-null  int64  
 11  Unsecured_TL                  42064 non-null  int64  
 12  Other_TL                      42064 non-null  int64  
 13  A

In [169]:
df_encoded.head()

,pct_tl_open_L6M,pct_tl_closed_L6M,Total_TL_opened_L12M,Tot_TL_closed_L12M,pct_tl_open_L12M,pct_tl_closed_L12M,Tot_Missed_Pmnt,CC_TL,Home_TL,PL_TL,...,last_prod_enq2_ConsumerLoan,last_prod_enq2_HL,last_prod_enq2_PL,last_prod_enq2_others,first_prod_enq2_AL,first_prod_enq2_CC,first_prod_enq2_ConsumerLoan,first_prod_enq2_HL,first_prod_enq2_PL,first_prod_enq2_others
0,0.000,0.0,0,0,0.00,0.000,0,0,0,4,...,0,0,1,0,0,0,0,0,1,0
1,0.000,0.0,1,0,1.00,0.000,0,0,0,0,...,1,0,0,0,0,0,1,0,0,0
2,0.125,0.0,2,0,0.25,0.000,1,0,0,0,...,1,0,0,0,0,0,0,0,0,1
3,0.000,0.0,0,0,0.00,0.000,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0.000,0.0,0,1,0.00,0.167,0,0,0,0,...,1,0,0,0,0,0,0,0,1,0


# Machine Learning model fitting

In [173]:
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )
y = df_encoded['Approved_Flag']

In [174]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [176]:
rf_classifier = RandomForestClassifier(n_estimators = 200, random_state=42)
rf_classifier.fit(x_train, y_train)
y_pred = rf_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print ()
print(f'Accuracy: {accuracy}')
print ()
precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()
    



Accuracy: 0.9895399976227267

Class p1:
Precision: 0.9314179796107507
Recall: 0.9911242603550295
F1 Score: 0.9603440038222647

Class p2:
Precision: 0.9986144101346002
Recall: 1.0
F1 Score: 0.9993067247697336

Class p3:
Precision: 0.994413407821229
Recall: 0.940377358490566
F1 Score: 0.9666408068269977

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [178]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

xgb_classifier = xgb.XGBClassifier(objective='multi:softmax',  num_class=4)



y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


x_train, x_test, y_train, y_test = train_test_split(x, y_encoded, test_size=0.2, random_state=42)




xgb_classifier.fit(x_train, y_train)
y_pred = xgb_classifier.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print ()
print(f'Accuracy: {accuracy:.2f}')
print ()

precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()




Accuracy: 0.99

Class p1:
Precision: 0.9868421052631579
Recall: 0.9615384615384616
F1 Score: 0.974025974025974

Class p2:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Class p3:
Precision: 0.9711324944485566
Recall: 0.990188679245283
F1 Score: 0.9805680119581465

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0



In [179]:
from sklearn.tree import DecisionTreeClassifier


y = df_encoded['Approved_Flag']
x = df_encoded. drop ( ['Approved_Flag'], axis = 1 )

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


dt_model = DecisionTreeClassifier(max_depth=20, min_samples_split=10)
dt_model.fit(x_train, y_train)
y_pred = dt_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print ()
print(f"Accuracy: {accuracy:.2f}")
print ()

precision, recall, f1_score, _ = precision_recall_fscore_support(y_test, y_pred)

for i, v in enumerate(['p1', 'p2', 'p3', 'p4']):
    print(f"Class {v}:")
    print(f"Precision: {precision[i]}")
    print(f"Recall: {recall[i]}")
    print(f"F1 Score: {f1_score[i]}")
    print()





Accuracy: 0.99

Class p1:
Precision: 0.9666666666666667
Recall: 0.9723865877712031
F1 Score: 0.9695181907571289

Class p2:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

Class p3:
Precision: 0.9787717968157695
Recall: 0.9743396226415094
F1 Score: 0.9765506807866868

Class p4:
Precision: 1.0
Recall: 1.0
F1 Score: 1.0

